In [1]:
# Import everything we need
from nba_api.stats.endpoints import playergamelog, leaguedashteamstats
from nba_api.stats.static import players
import pandas as pd
import numpy as np
import time

print("All imports successful!")
print("Today we build our first ML model.")
print("By the end of this notebook we will predict LeBron's scoring in any game.")

All imports successful!
Today we build our first ML model.
By the end of this notebook we will predict LeBron's scoring in any game.


In [2]:
# Step 1: Fetch all the data we need
# We learned this in Days 1 and 2, now we are putting it all together

time.sleep(1)

# Fetch LeBron's game logs
print("Fetching LeBron's game logs...")
gamelog = playergamelog.PlayerGameLog(player_id=2544, season='2024-25')
games = gamelog.get_data_frames()[0]

time.sleep(1)

# Fetch all team defensive stats
print("Fetching team defensive stats...")
team_stats = leaguedashteamstats.LeagueDashTeamStats(
    season='2024-25',
    measure_type_detailed_defense='Defense'
)
defense_df = team_stats.get_data_frames()[0]

print(f"\nGames fetched: {len(games)}")
print(f"Teams fetched: {len(defense_df)}")
print("\nRaw data ready. Now building features...")

Fetching LeBron's game logs...
Fetching team defensive stats...

Games fetched: 70
Teams fetched: 30

Raw data ready. Now building features...


In [3]:
# Step 2: Feature Engineering
# This is where we transform raw data into ML-ready features
# Think of this as preparing ingredients before cooking

# Team abbreviation to full name mapping
abbrev_to_name = {
    'ATL': 'Atlanta Hawks', 'BOS': 'Boston Celtics', 'BKN': 'Brooklyn Nets',
    'CHA': 'Charlotte Hornets', 'CHI': 'Chicago Bulls', 'CLE': 'Cleveland Cavaliers',
    'DAL': 'Dallas Mavericks', 'DEN': 'Denver Nuggets', 'DET': 'Detroit Pistons',
    'GSW': 'Golden State Warriors', 'HOU': 'Houston Rockets', 'IND': 'Indiana Pacers',
    'LAC': 'LA Clippers', 'LAL': 'Los Angeles Lakers', 'MEM': 'Memphis Grizzlies',
    'MIA': 'Miami Heat', 'MIL': 'Milwaukee Bucks', 'MIN': 'Minnesota Timberwolves',
    'NOP': 'New Orleans Pelicans', 'NYK': 'New York Knicks', 'OKC': 'Oklahoma City Thunder',
    'ORL': 'Orlando Magic', 'PHI': 'Philadelphia 76ers', 'PHX': 'Phoenix Suns',
    'POR': 'Portland Trail Blazers', 'SAC': 'Sacramento Kings', 'SAS': 'San Antonio Spurs',
    'TOR': 'Toronto Raptors', 'UTA': 'Utah Jazz', 'WAS': 'Washington Wizards'
}

# Sort oldest to newest for rolling calculations
games['GAME_DATE'] = pd.to_datetime(games['GAME_DATE'])
games = games.sort_values('GAME_DATE').reset_index(drop=True)

# Feature 1: Home or Away
# 1 = Home, 0 = Away
games['IS_HOME'] = games['MATCHUP'].apply(lambda x: 1 if 'vs.' in x else 0)

# Feature 2: Days of rest
games['DAYS_REST'] = games['GAME_DATE'].diff().dt.days.fillna(3)

# Feature 3: Opponent defensive rating
defense_lookup = dict(zip(defense_df['TEAM_NAME'], defense_df['DEF_RATING']))
games['OPPONENT'] = games['MATCHUP'].apply(lambda x: x.split()[-1])
games['OPPONENT_NAME'] = games['OPPONENT'].map(abbrev_to_name)
games['OPP_DEF_RATING'] = games['OPPONENT_NAME'].map(defense_lookup)

# Feature 4: Recent form (last 5 game average points)
# shift(1) means we use PREVIOUS games only, not the current game
# This is critical, we cannot use current game data to predict current game
games['FORM_5'] = games['PTS'].rolling(5).mean().shift(1)

# Feature 5: Team momentum (win rate last 5 games)
games['WIN'] = games['WL'].apply(lambda x: 1 if x == 'W' else 0)
games['WIN_RATE_5'] = games['WIN'].rolling(5).mean().shift(1)

# Target variable: what we are trying to predict
# This is the answer our model is trying to learn
target = 'PTS'

print("Features engineered successfully!")
print(f"\nDataset shape before cleaning: {games.shape}")

# Drop rows where we don't have enough history for rolling features
# The first 5 games won't have FORM_5 or WIN_RATE_5 values
games_clean = games.dropna(subset=['FORM_5', 'WIN_RATE_5', 'OPP_DEF_RATING'])
print(f"Dataset shape after cleaning: {games_clean.shape}")
print(f"\nRows removed: {len(games) - len(games_clean)} (not enough history for rolling features)")

Features engineered successfully!

Dataset shape before cleaning: (70, 35)
Dataset shape after cleaning: (65, 35)

Rows removed: 5 (not enough history for rolling features)


In [5]:
# Step 3: Prepare the data for scikit-learn
# scikit-learn expects data in a very specific format
# X = features (the inputs, what we know before the game)
# y = target (the output, what we are trying to predict)

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# Select only our 5 features
feature_columns = [
    'IS_HOME',        # Feature 1: home or away
    'DAYS_REST',      # Feature 2: days of rest
    'OPP_DEF_RATING', # Feature 3: opponent defense strength
    'FORM_5',         # Feature 4: recent scoring form
    'WIN_RATE_5'      # Feature 5: team momentum
]

X = games_clean[feature_columns]
y = games_clean['PTS']

print("=== Our Training Dataset ===")
print(f"Number of games (samples): {len(X)}")
print(f"Number of features: {len(feature_columns)}")
print(f"\nFeature statistics:")
print(X.describe().round(2))
print(f"\nTarget (PTS) statistics:")
print(y.describe().round(1))

=== Our Training Dataset ===
Number of games (samples): 65
Number of features: 5

Feature statistics:
       IS_HOME  DAYS_REST  OPP_DEF_RATING  FORM_5  WIN_RATE_5
count    65.00      65.00           65.00   65.00       65.00
mean      0.48       2.51          113.57   24.68        0.61
std       0.50       2.01            3.39    3.38        0.22
min       0.00       1.00          106.60   16.60        0.20
25%       0.00       2.00          111.00   22.60        0.40
50%       0.00       2.00          113.60   24.60        0.60
75%       1.00       2.00          115.70   27.00        0.80
max       1.00      14.00          119.40   31.20        1.00

Target (PTS) statistics:
count    65.0
mean     24.7
std       7.3
min      10.0
25%      19.0
50%      25.0
75%      29.0
max      42.0
Name: PTS, dtype: float64


In [6]:
# Step 4: Split data into training and testing sets
# This is a critical concept in machine learning
# 
# Analogy: Imagine you are studying for an exam
# You study using practice questions (training set)
# Then you take the real exam with questions you have never seen (test set)
# If you only memorized the practice questions you will fail the real exam
# A good student actually understands the material and can answer new questions
# A good ML model actually learns patterns and can predict new games
#
# We use 80% of games to train and 20% to test
# test_size=0.2 means 20% goes to testing
# random_state=42 means the split is reproducible every time we run it

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training games: {len(X_train)} (80%)")
print(f"Testing games:  {len(X_test)} (20%)")
print("\nThink of it this way:")
print(f"  We show the model {len(X_train)} games to learn from")
print(f"  Then we test it on {len(X_test)} games it has never seen")
print("\nNow training the model...")

# Step 5: Train the model
# This is the moment where the magic happens
# LinearRegression().fit() finds the best weights for each feature
# It looks at all 52 training games and figures out:
# "How much does home court matter? How much does form matter?"
# This takes less than a second because 65 games is a small dataset

model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully!")
print("\nThe model has learned the following weights:")
for feature, weight in zip(feature_columns, model.coef_):
    print(f"  {feature}: {weight:.3f}")
print(f"  Base score: {model.intercept_:.3f}")

Training games: 52 (80%)
Testing games:  13 (20%)

Think of it this way:
  We show the model 52 games to learn from
  Then we test it on 13 games it has never seen

Now training the model...
Model trained successfully!

The model has learned the following weights:
  IS_HOME: 1.573
  DAYS_REST: -0.845
  OPP_DEF_RATING: 0.003
  FORM_5: -0.107
  WIN_RATE_5: 6.279
  Base score: 24.599


In [7]:
# Step 6: Test the model on games it has never seen
# This is the moment of truth
# We feed it the 13 test games and see how close its predictions are

y_predicted = model.predict(X_test)

# Mean Absolute Error = on average how many points off are we?
# If MAE is 5 it means our predictions are off by 5 points on average
mae = mean_absolute_error(y_test, y_predicted)

# R2 Score = how much of the variation does our model explain?
# 1.0 = perfect prediction, 0.0 = no better than just guessing the average
# 0.5 means our model explains 50% of the variation in scoring
r2 = r2_score(y_test, y_predicted)

print("=== Model Performance on Unseen Games ===\n")
print(f"Mean Absolute Error: {mae:.1f} points")
print(f"R2 Score: {r2:.3f}")
print(f"\nIn plain English:")
print(f"Our model predicts LeBron's scoring within {mae:.1f} points on average")
print(f"It explains {r2*100:.1f}% of the variation in his scoring")

print("\n=== Predicted vs Actual for Each Test Game ===")
results = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_predicted.round(1),
    'Error': abs(y_test.values - y_predicted).round(1)
})
print(results.to_string())
print(f"\nAverage error: {results['Error'].mean().round(1)} points")

=== Model Performance on Unseen Games ===

Mean Absolute Error: 4.4 points
R2 Score: 0.138

In plain English:
Our model predicts LeBron's scoring within 4.4 points on average
It explains 13.8% of the variation in his scoring

=== Predicted vs Actual for Each Test Game ===
    Actual  Predicted  Error
0       17       16.9    0.1
1       27       27.3    0.3
2       27       24.8    2.2
3       25       23.8    1.2
4       35       25.1    9.9
5       19       24.5    5.5
6       29       24.0    5.0
7       16       24.2    8.2
8       14       25.7   11.7
9       29       25.0    4.0
10      20       26.1    6.1
11      31       28.6    2.4
12      24       24.9    0.9

Average error: 4.4 points


In [10]:
import warnings
warnings.filterwarnings('ignore')
# Step 7: Build a prediction function
# This is what will power our SportIQ app
# You give it game conditions and it predicts the score

def predict_lebron_score(is_home, days_rest, opp_def_rating, recent_form, win_rate):
    """
    Predict LeBron's scoring given game conditions.
    
    Parameters:
    is_home: 1 if home game, 0 if away
    days_rest: number of days since last game
    opp_def_rating: opponent's defensive rating (lower = tougher defense)
    recent_form: LeBron's average points in last 5 games
    win_rate: Lakers win rate in last 5 games (0.0 to 1.0)
    
    Returns:
    Predicted points (rounded to 1 decimal)
    """
    features = [[is_home, days_rest, opp_def_rating, recent_form, win_rate]]
    prediction = model.predict(features)[0]
    return round(prediction, 1)

# Test it with some realistic scenarios
print("=== SportIQ Prediction Engine ===\n")

print("Scenario 1: LeBron at home vs Utah Jazz (worst defense)")
print("  Conditions: Home, 2 days rest, Utah DEF_RATING=119.4, form=25 PPG, win rate=0.8")
pred1 = predict_lebron_score(1, 2, 119.4, 25, 0.8)
print(f"  Predicted score: {pred1} points\n")

print("Scenario 2: LeBron away vs OKC Thunder (best defense)")
print("  Conditions: Away, 1 day rest, OKC DEF_RATING=106.6, form=22 PPG, win rate=0.4")
pred2 = predict_lebron_score(0, 1, 106.6, 22, 0.4)
print(f"  Predicted score: {pred2} points\n")

print("Scenario 3: LeBron at home, hot streak, playing weak team")
print("  Conditions: Home, 2 days rest, DEF_RATING=117.0, form=30 PPG, win rate=1.0")
pred3 = predict_lebron_score(1, 2, 117.0, 30, 1.0)
print(f"  Predicted score: {pred3} points\n")

print("Scenario 4: LeBron away, cold streak, playing tough defense")
print("  Conditions: Away, 3 days rest, DEF_RATING=108.0, form=18 PPG, win rate=0.2")
pred4 = predict_lebron_score(0, 3, 108.0, 18, 0.2)
print(f"  Predicted score: {pred4} points")

=== SportIQ Prediction Engine ===

Scenario 1: LeBron at home vs Utah Jazz (worst defense)
  Conditions: Home, 2 days rest, Utah DEF_RATING=119.4, form=25 PPG, win rate=0.8
  Predicted score: 27.2 points

Scenario 2: LeBron away vs OKC Thunder (best defense)
  Conditions: Away, 1 day rest, OKC DEF_RATING=106.6, form=22 PPG, win rate=0.4
  Predicted score: 24.3 points

Scenario 3: LeBron at home, hot streak, playing weak team
  Conditions: Home, 2 days rest, DEF_RATING=117.0, form=30 PPG, win rate=1.0
  Predicted score: 27.9 points

Scenario 4: LeBron away, cold streak, playing tough defense
  Conditions: Away, 3 days rest, DEF_RATING=108.0, form=18 PPG, win rate=0.2
  Predicted score: 21.7 points


In [11]:
import joblib
import os

# Create a models folder to store our trained models
os.makedirs('../models', exist_ok=True)

# Save the model to disk
# joblib is like pickle but optimized for numpy arrays and sklearn models
# Think of it like saving a Word document so you can open it later
joblib.dump(model, '../models/lebron_scorer.pkl')

# Save the feature column names too so we know what order to feed features
joblib.dump(feature_columns, '../models/feature_columns.pkl')

print("Model saved successfully!")
print("Files created:")
print("  models/lebron_scorer.pkl      - the trained ML model")
print("  models/feature_columns.pkl   - the feature column names")
print("\nThis model can now be loaded by our FastAPI backend")
print("and used to make predictions in real time!")

Model saved successfully!
Files created:
  models/lebron_scorer.pkl      - the trained ML model
  models/feature_columns.pkl   - the feature column names

This model can now be loaded by our FastAPI backend
and used to make predictions in real time!
